In [1]:
# Cell 1: Imports and Path Setup
import sys
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict, field
import pickle

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd() / "workspace" / "trade-automation"))
os.chdir(Path.cwd() / "workspace" / "trade-automation")

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# LSTM modules
from crypto_analysis.lstm.model import ModelConfig, CNNLSTMSignalPredictor
from crypto_analysis.lstm.trainer import Trainer, TrainingConfig, TrainingHistory
from crypto_analysis.lstm.loss import BinarySignalLoss, FocalBinaryLoss
from crypto_analysis.lstm.dataset import SignalDataset, create_sequences

# VectorBT Data Preprocessor (replaces DataPreprocessor and DatasetBuilder)
from crypto_analysis.vectorbt_optimizer import VectorBTDataPreprocessor

# MLflow
import mlflow
import mlflow.pytorch

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MLflow version: {mlflow.__version__}")

PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3060
MLflow version: 3.8.1


In [2]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 638 bytes | 106.00 KiB/s, done.
From https://github.com/bayazknn/trade-automation
   2f4553a..5f1a01a  main       -> origin/main
Updating 2f4553a..5f1a01a
Fast-forward
 crypto_analysis/lstm_optimizer.py | 18 +++++++++++++++---
 1 file changed, 15 insertions(+), 3 deletions(-)


In [2]:
# Cell 2: Configuration Constants

# === CONFIGURATION ===

# Data paths - VectorBT optimized CSV files
CSV_DIR = Path("notebooks/csvs")  # Directory with vectorbt_optimizer output CSVs
OUTPUT_DIR = Path("notebooks/coin_csvs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

# VectorBTDataPreprocessor configuration
PREPROCESSOR_CONFIG = {
    "remove_raw_indicators": False,   # Keep only OHLCV + *_entry/*_exit signals
    "target_shift": 1,               # Features at t predict target at t+1
    "sequence_length": 18,           # LSTM input sequence length
    "stride": 1,                     # Step between sequences
    "scaler_type": "standard",       # 'standard' or 'minmax'
    "target_column": "tradeable",
}

# Train/val/test split ratios
SPLIT_CONFIG = {
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "test_ratio": 0.2,
}

# MLflow experiment name
MLFLOW_EXPERIMENT = "multi_coin_lstm_training"

print("Configuration loaded.")
print(f"CSV directory: {CSV_DIR.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Sequence length: {PREPROCESSOR_CONFIG['sequence_length']}")
print(f"Target shift: {PREPROCESSOR_CONFIG['target_shift']}")
print(f"Remove raw indicators: {PREPROCESSOR_CONFIG['remove_raw_indicators']}")

Configuration loaded.
CSV directory: /workspace/trade-automation/notebooks/csvs
Output directory: /workspace/trade-automation/notebooks/coin_csvs
Sequence length: 18
Target shift: 1
Remove raw indicators: False


In [3]:
# from crypto_analysis.vectorbt_optimizer import optimize_all
# results = optimize_all(
#     symbols="DOGE",
#     indicators="all",
#     data_dir="data/binance",
#     output_dir="notebooks/test_csv",
#     config_path="config.json",
#     n_processes=1,
#     n_jobs_optuna=4,
#     threshold_pct=2,
#     period_hours=6,
#     export_csv=True,
#     export_params_json=True
# )

In [4]:
preprocessor = VectorBTDataPreprocessor(
    remove_raw_indicators=PREPROCESSOR_CONFIG["remove_raw_indicators"],
    target_shift=PREPROCESSOR_CONFIG["target_shift"],
    sequence_length=PREPROCESSOR_CONFIG["sequence_length"],
    stride=PREPROCESSOR_CONFIG["stride"],
    scaler_type=PREPROCESSOR_CONFIG["scaler_type"],
    target_column=PREPROCESSOR_CONFIG["target_column"],
)

results = preprocessor.load_csv("notebooks/test_csv/DOGE_optimized.csv")
preprocessor.fit(results)
print(f"\nAfter fitting:")
print(preprocessor)
print(f"\nFeature columns: {preprocessor.get_feature_names()}")
print(f"Number of features: {preprocessor.get_num_features()}")

# Store for later use
n_features = preprocessor.get_num_features()
feature_names = preprocessor.get_feature_names()


After fitting:
VectorBTDataPreprocessor(sequence_length=18, target_shift=1, stride=1, remove_raw_indicators=False, n_features=185, n_ohlcv=5, n_signals=180, scaler_type='standard')

Feature columns: ['open', 'high', 'low', 'close', 'volume', 'RSI_entry', 'RSI_exit', 'RSI_rsi', 'STOCH_entry', 'STOCH_exit', 'STOCH_slowk', 'STOCH_slowd', 'STOCHRSI_entry', 'STOCHRSI_exit', 'STOCHRSI_fastk', 'STOCHRSI_fastd', 'CCI_entry', 'CCI_exit', 'CCI_cci', 'MFI_entry', 'MFI_exit', 'MFI_mfi', 'WILLR_entry', 'WILLR_exit', 'WILLR_willr', 'CMO_entry', 'CMO_exit', 'CMO_cmo', 'ADX_entry', 'ADX_exit', 'ADX_adx', 'MOM_entry', 'MOM_exit', 'MOM_mom', 'ROC_entry', 'ROC_exit', 'ROC_roc', 'TRIX_entry', 'TRIX_exit', 'TRIX_trix', 'ULTOSC_entry', 'ULTOSC_exit', 'ULTOSC_ultosc', 'APO_entry', 'APO_exit', 'APO_apo', 'PPO_entry', 'PPO_exit', 'PPO_ppo', 'BOP_entry', 'BOP_exit', 'BOP_bop', 'AROON_entry', 'AROON_exit', 'AROON_aroondown', 'AROON_aroonup', 'AROONOSC_entry', 'AROONOSC_exit', 'AROONOSC_aroonosc', 'ADXR_entry', 

In [5]:
# Cell 5: Create Sequences for Each Coin (Separately for Proper Splitting)

def create_coin_sequences_separate(
    preprocessor: VectorBTDataPreprocessor,
    dataframes: Dict[str, pd.DataFrame],
    verbose: bool = True
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Create sequences for each coin separately.
    
    This allows us to split each coin's data before concatenating,
    ensuring no data leakage between train/val/test sets.
    
    Parameters
    ----------
    preprocessor : VectorBTDataPreprocessor
        Fitted preprocessor
    dataframes : Dict[str, pd.DataFrame]
        Symbol -> DataFrame mapping
    verbose : bool
        Print progress
    
    Returns
    -------
    Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (X sequences, y sequences) mapping
    """
    sequences = {}
    
    if verbose:
        print("Creating sequences for each coin...")
        print(f"  Sequence length: {preprocessor.sequence_length}")
        print(f"  Target shift: {preprocessor.target_shift}")
        print(f"  Stride: {preprocessor.stride}")
        print("-" * 60)
    
    for name, df in dataframes.items():
        try:
            # Transform to features/targets
            features, targets = preprocessor.fit_transform(df)
            
            # Create sequences
            X, y = preprocessor.create_sequences(features, targets)
            sequences[name] = (X, y)
            
            if verbose:
                trade_pct = (y == 1).mean() * 100
                print(f"  {name}: X={X.shape}, y={y.shape} | Trade: {trade_pct:.1f}%")
                
        except Exception as e:
            print(f"  {name}: Error - {e}")
    
    if verbose:
        print("-" * 60)
        total_seqs = sum(X.shape[0] for X, y in sequences.values())
        print(f"Total sequences: {total_seqs}")
    
    return sequences


# Create sequences for each coin

coin_sequences = create_coin_sequences_separate(preprocessor, {"DOGE":results}, verbose=True)


Creating sequences for each coin...
  Sequence length: 18
  Target shift: 1
  Stride: 1
------------------------------------------------------------
  DOGE: X=(8889, 18, 185), y=(8889,) | Trade: 41.5%
------------------------------------------------------------
Total sequences: 8889


In [6]:
# Cell 7: Dataset Splitting and Concatenation Utilities

def split_coin_sequences(
    sequences: Dict[str, Tuple[np.ndarray, np.ndarray]],
    split_config: Dict
) -> Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]:
    """
    Split each coin's sequences into train/val/test sets.
    Uses sequential split (no shuffle) to preserve temporal order.
    
    Parameters
    ----------
    sequences : Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (X, y) mapping
    split_config : Dict
        Split ratios (train_ratio, val_ratio, test_ratio)
    
    Returns
    -------
    Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]
        Symbol -> {'train': (X, y), 'val': (X, y), 'test': (X, y)}
    """
    split_data = {}
    
    train_ratio = split_config["train_ratio"]
    val_ratio = split_config["val_ratio"]
    
    print(f"Splitting sequences: {train_ratio*100:.0f}% train, {val_ratio*100:.0f}% val, "
          f"{split_config['test_ratio']*100:.0f}% test")
    print("-" * 70)
    
    for name, (X, y) in sequences.items():
        n = len(X)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        split_data[name] = {
            'train': (X[:train_end], y[:train_end]),
            'val': (X[train_end:val_end], y[train_end:val_end]),
            'test': (X[val_end:], y[val_end:])
        }
        
        print(f"{name}: train={train_end}, val={val_end - train_end}, test={n - val_end}")
    
    return split_data


def concatenate_coin_datasets(
    split_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]],
    device: Optional[torch.device] = None
) -> Dict[str, SignalDataset]:
    """
    Concatenate all coins' train/val/test sets into unified datasets.
    
    Parameters
    ----------
    split_data : Dict
        Split data from split_coin_sequences
    device : torch.device, optional
        Device for tensors
    
    Returns
    -------
    Dict[str, SignalDataset]
        {'train': SignalDataset, 'val': SignalDataset, 'test': SignalDataset}
    """
    combined = {
        'train': {'X': [], 'y': []},
        'val': {'X': [], 'y': []},
        'test': {'X': [], 'y': []}
    }
    
    for name, splits in split_data.items():
        for split_name in ['train', 'val', 'test']:
            X, y = splits[split_name]
            combined[split_name]['X'].append(X)
            combined[split_name]['y'].append(y)
    
    datasets = {}
    print("\n" + "=" * 50)
    print("CONCATENATED DATASETS")
    print("=" * 50)
    
    for split_name in ['train', 'val', 'test']:
        X_concat = np.concatenate(combined[split_name]['X'], axis=0)
        y_concat = np.concatenate(combined[split_name]['y'], axis=0)
        datasets[split_name] = SignalDataset(X_concat, y_concat, device=device)
        
        dist = datasets[split_name].get_class_distribution()
        print(f"{split_name:5s}: {len(datasets[split_name]):6d} samples | "
              f"Hold: {dist['hold']:5d} | Trade: {dist['trade']:5d}")
    
    print("=" * 50)
    return datasets


def analyze_dataset_distribution(
    datasets: Dict[str, SignalDataset]
) -> Dict[str, Dict]:
    """
    Analyze class distribution in each dataset split.
    
    Parameters
    ----------
    datasets : Dict[str, SignalDataset]
        Train/val/test datasets
    
    Returns
    -------
    Dict[str, Dict]
        Distribution statistics for each split
    """
    distributions = {}
    
    print("\n" + "=" * 70)
    print("DATASET DISTRIBUTION ANALYSIS")
    print("=" * 70)
    
    for split_name, dataset in datasets.items():
        dist = dataset.get_class_distribution()
        total = dist['hold'] + dist['trade']
        hold_pct = dist['hold'] / total * 100 if total > 0 else 0
        trade_pct = dist['trade'] / total * 100 if total > 0 else 0
        imbalance_ratio = dist['hold'] / max(dist['trade'], 1)
        
        distributions[split_name] = {
            'total': total,
            'hold': dist['hold'],
            'trade': dist['trade'],
            'hold_pct': hold_pct,
            'trade_pct': trade_pct,
            'imbalance_ratio': imbalance_ratio
        }
        
        print(f"\n{split_name.upper()}:")
        print(f"  Total samples: {total}")
        print(f"  Hold:  {dist['hold']:6d} ({hold_pct:5.2f}%)")
        print(f"  Trade: {dist['trade']:6d} ({trade_pct:5.2f}%)")
        print(f"  Imbalance ratio (hold/trade): {imbalance_ratio:.2f}")
    
    print("\n" + "=" * 70)
    return distributions

In [7]:
# Cell 8: Split and Concatenate Datasets

# Split each coin's sequences
split_coin_data = split_coin_sequences(coin_sequences, SPLIT_CONFIG)

# Concatenate into unified datasets
multi_coin_datasets = concatenate_coin_datasets(split_coin_data)

# Analyze distribution
dataset_distributions = analyze_dataset_distribution(multi_coin_datasets)

Splitting sequences: 60% train, 20% val, 20% test
----------------------------------------------------------------------
DOGE: train=5333, val=1778, test=1778

CONCATENATED DATASETS
train:   5333 samples | Hold:  2834 | Trade:  2499
val  :   1778 samples | Hold:  1129 | Trade:   649
test :   1778 samples | Hold:  1235 | Trade:   543

DATASET DISTRIBUTION ANALYSIS

TRAIN:
  Total samples: 5333
  Hold:    2834 (53.14%)
  Trade:   2499 (46.86%)
  Imbalance ratio (hold/trade): 1.13

VAL:
  Total samples: 1778
  Hold:    1129 (63.50%)
  Trade:    649 (36.50%)
  Imbalance ratio (hold/trade): 1.74

TEST:
  Total samples: 1778
  Hold:    1235 (69.46%)
  Trade:    543 (30.54%)
  Imbalance ratio (hold/trade): 2.27



In [8]:
# # Cell 9: Create Model Configuration

# model_config = ModelConfig(
#     input_size=n_features,
#     hidden_size=128,
#     num_layers=2,
#     dropout=0.1,
#     bidirectional=False,
#     num_classes=2,
#     input_seq_length=PREPROCESSOR_CONFIG["sequence_length"],
#     classifier_hidden_size=32,
#     # CNN-LSTM specific
#     kernel_size=5,
#     cnn_num_layers=3,
#     cnn_dropout=0.1,
#     lstm_dropout=0.1,
#     classifier_dropout=0.1,
# )

# print("Model Configuration:")
# print("=" * 40)
# for field_name, value in asdict(model_config).items():
#     print(f"  {field_name}: {value}")

In [9]:
# # Cell 10: Create Model

# model = CNNLSTMSignalPredictor(model_config)
# print(model)
# print(f"\nTotal trainable parameters: {model.get_num_parameters():,}")

In [10]:
# # Cell 12: Create Training Configuration

# training_config = TrainingConfig(
#     # Model architecture (for reference)
#     hidden_size=model_config.hidden_size,
#     num_layers=model_config.num_layers,
#     dropout=model_config.dropout,
    
#     # Training parameters
#     epochs=200,
#     batch_size=32,
#     learning_rate=5e-3,
#     weight_decay=1e-3,
#     optimizer='adamw',
#     grad_clip_norm=1.0,
    
#     # Learning rate scheduler
#     scheduler='plateau',
#     scheduler_patience=20,
#     scheduler_factor=0.8,
    
#     # Class imbalance handling (from helper function)
#     auto_class_weights=True,
#     class_weight_power=0.5,
#     focal_loss=False,
#     focal_gamma=2.0,
#     label_smoothing=0.1,
    
#     # Data split (we provide pre-split datasets)
#     val_split=0.2,
#     test_split=0.2,
    
#     # Early stopping
#     early_stopping=False,
#     patience=20,
#     min_delta=1e-4,
    
#     # Checkpointing
#     checkpoint_dir=str(OUTPUT_DIR / "checkpoints"),
#     save_best_only=True,
    
#     # Device
#     device='auto',
    
#     # Logging
#     log_interval=50,
#     verbose=False,
# )

# print("Training Configuration created.")
# print(f"Epochs: {training_config.epochs}")
# print(f"Batch size: {training_config.batch_size}")
# print(f"Learning rate: {training_config.learning_rate}")
# print(f"Focal loss: {training_config.focal_loss}")
# print(f"Class weight power: {training_config.class_weight_power}")

In [11]:
# Cell 13: MLflow Setup and Logging Functions

def setup_mlflow(experiment_name: str) -> str:
    """
    Setup MLflow experiment.
    
    Parameters
    ----------
    experiment_name : str
        Name for the MLflow experiment
    
    Returns
    -------
    str
        Experiment ID
    """
    # Set tracking URI (default is local ./mlruns)
    mlflow.set_tracking_uri("http://localhost:5000")
    
    # Create or get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
    else:
        experiment_id = experiment.experiment_id
    
    mlflow.set_experiment(experiment_name)
    
    print(f"MLflow experiment: {experiment_name}")
    print(f"Experiment ID: {experiment_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    
    return experiment_id


def log_training_params(
    model_config: ModelConfig,
    training_config: TrainingConfig,
    preprocessor_config: Dict,
    symbols: List[str],
    distributions: Dict[str, Dict]
):
    """
    Log all configuration parameters to MLflow.
    """
    # Model params
    mlflow.log_params({
        "model.input_size": model_config.input_size,
        "model.hidden_size": model_config.hidden_size,
        "model.num_layers": model_config.num_layers,
        "model.dropout": model_config.dropout,
        "model.bidirectional": model_config.bidirectional,
        "model.classifier_hidden_size": model_config.classifier_hidden_size,
        "model.kernel_size": model_config.kernel_size,
        "model.cnn_num_layers": model_config.cnn_num_layers,
        "model.cnn_dropout": model_config.cnn_dropout,
        "model.lstm_dropout": model_config.lstm_dropout,
    })
    
    # Training params
    mlflow.log_params({
        "train.epochs": training_config.epochs,
        "train.batch_size": training_config.batch_size,
        "train.learning_rate": training_config.learning_rate,
        "train.weight_decay": training_config.weight_decay,
        "train.optimizer": training_config.optimizer,
        "train.scheduler": training_config.scheduler,
        "train.focal_loss": training_config.focal_loss,
        "train.focal_gamma": training_config.focal_gamma,
        "train.class_weight_power": training_config.class_weight_power,
        "train.label_smoothing": training_config.label_smoothing,
        "train.early_stopping": training_config.early_stopping,
        "train.patience": training_config.patience,
    })
    
    # Preprocessor params
    mlflow.log_params({
        "data.sequence_length": preprocessor_config["sequence_length"],
        "data.target_shift": preprocessor_config["target_shift"],
        "data.stride": preprocessor_config["stride"],
        "data.remove_raw_indicators": preprocessor_config["remove_raw_indicators"],
        "data.num_coins": len(symbols),
    })
    
    # Distribution info
    train_dist = distributions.get('train', {})
    mlflow.log_params({
        "dist.train_samples": train_dist.get('total', 0),
        "dist.train_trade_pct": round(train_dist.get('trade_pct', 0), 2),
        "dist.imbalance_ratio": round(train_dist.get('imbalance_ratio', 1), 2),
    })


def log_training_metrics(history: TrainingHistory, epoch: int):
    """
    Log training metrics for a single epoch.
    """
    mlflow.log_metrics({
        "train_loss": history.train_losses[-1],
        "val_loss": history.val_losses[-1],
        "train_accuracy": history.train_accuracies[-1],
        "val_accuracy": history.val_accuracies[-1],
        "learning_rate": history.learning_rates[-1],
    }, step=epoch)


def log_evaluation_results(results: Dict[str, Dict]):
    """
    Log final evaluation metrics.
    """
    for split_name, metrics in results.items():
        if metrics is None:
            continue
        prefix = f"{split_name}_"
        mlflow.log_metrics({
            f"{prefix}accuracy": metrics['accuracy'],
            f"{prefix}precision": metrics['precision'],
            f"{prefix}recall": metrics['recall'],
            f"{prefix}f1": metrics['f1'],
            f"{prefix}hold_f1": metrics['hold_f1'],
            f"{prefix}trade_f1": metrics['trade_f1'],
        })


def log_model_artifact(model: torch.nn.Module, preprocessor: VectorBTDataPreprocessor, output_dir: Path):
    """
    Log model and preprocessor as MLflow artifacts.
    """
    # Log PyTorch model
    mlflow.pytorch.log_model(model, "model")
    
    # Log preprocessor
    preprocessor_path = output_dir / "preprocessor_mlflow.pkl"
    preprocessor.save(preprocessor_path)
    mlflow.log_artifact(str(preprocessor_path))
    preprocessor_path.unlink()  # Clean up temp file

In [12]:
# experiment_id = setup_mlflow(MLFLOW_EXPERIMENT)

In [13]:
# # Cell 15: Training Function with MLflow

# def train_with_mlflow(
#     model: torch.nn.Module,
#     training_config: TrainingConfig,
#     train_dataset: SignalDataset,
#     val_dataset: SignalDataset,
#     test_dataset: SignalDataset,
#     model_config: ModelConfig,
#     preprocessor_config: Dict,
#     symbols: List[str],
#     distributions: Dict[str, Dict],
#     preprocessor: VectorBTDataPreprocessor,
#     output_dir: Path
# ) -> Tuple[TrainingHistory, Dict, Trainer, str]:
#     """
#     Train model with full MLflow tracking.
    
#     Returns
#     -------
#     Tuple[TrainingHistory, Dict, Trainer, str]
#         (history, evaluation_results, trainer, run_id)
#     """
#     with mlflow.start_run() as run:
#         run_id = run.info.run_id
#         print(f"MLflow Run ID: {run_id}")
#         print("=" * 60)
        
#         # Log all parameters
#         log_training_params(
#             model_config, training_config, 
#             preprocessor_config,
#             symbols, distributions
#         )
        
#         # Create trainer (use preprocessor for consistency, though VectorBTDataPreprocessor
#         # doesn't have the same interface as DataPreprocessor - we'll pass None)
#         trainer = Trainer(
#             model=model,
#             config=training_config,
#             preprocessor=None  # VectorBTDataPreprocessor has different interface
#         )
        
#         # Store test dataset for evaluation
#         trainer.test_dataset = test_dataset
        
#         # Define callback for epoch-level logging
#         def mlflow_callback(epoch: int, history: TrainingHistory):
#             log_training_metrics(history, epoch)
        
#         # Train
#         print("\nStarting training...")
#         history = trainer.train(
#             train_dataset=train_dataset,
#             val_dataset=val_dataset,
#             callbacks=[mlflow_callback]
#         )
        
#         # Evaluate
#         print("\nEvaluating model...")
#         results = trainer.evaluate_all(verbose=True)
        
#         # Log evaluation metrics
#         log_evaluation_results(results)
        
#         # Log final metrics
#         mlflow.log_metrics({
#             "best_epoch": history.best_epoch + 1,
#             "best_val_loss": history.best_val_loss,
#             "epochs_trained": len(history.train_losses),
#         })
        
#         # Log model artifact
#         log_model_artifact(model, preprocessor, output_dir)
        
#         print(f"\nMLflow Run completed: {run_id}")
        
#     return history, results, trainer, run_id

In [14]:

# # Get symbols list
# symbols = list(results.keys())

# # Execute training with MLflow
# history, eval_results, trainer, run_id = train_with_mlflow(
#     model=model,
#     training_config=training_config,
#     train_dataset=multi_coin_datasets['train'],
#     val_dataset=multi_coin_datasets['val'],
#     test_dataset=multi_coin_datasets['test'],
#     model_config=model_config,
#     preprocessor_config=PREPROCESSOR_CONFIG,
#     symbols=symbols,
#     distributions=dataset_distributions,
#     preprocessor=preprocessor,
#     output_dir=OUTPUT_DIR
# )

# print(f"\nTraining complete!")
# print(f"Best epoch: {history.best_epoch + 1}")
# print(f"Best validation loss: {history.best_val_loss:.4f}")
# print(f"MLflow Run ID: {run_id}")

In [15]:
from crypto_analysis.lstm_optimizer import LSTMMetaheuristicOptimizer

In [ ]:
optimizer = LSTMMetaheuristicOptimizer(
    df=results,  # Use DOGE data for optimization
    preprocessor_type='vectorbt',  # Use new VectorBT preprocessor
    enable_mlflow=True,
    mlflow_experiment_name='lstm_crypto_optimization',
    mlflow_tracking_uri='http://127.0.0.1:5000',  # Optional
    pop_size=15,
    iterations=70,
    n_workers=15,
    np_neighbors=2,
    pf_max=0.2,
    elitist_selection=False,
    model_type="cnn_lstm",
    epochs_per_eval=125,
)
result = optimizer.optimize()


LSTMMetaheuristicOptimizer (APO) initialized:
  - Model type: cnn_lstm
  - Preprocessor type: vectorbt
  - DataFrame mode: binary
  - Feature columns: 196
  - Hyperparameters: 14
  - Total dimension: 210
  - Population size: 15
  - Iterations: 70
  - Workers: 15
  - APO np_neighbors: 2
  - APO pf_max: 0.2
  - Feature selection: True
  - Elitist selection: False
  - MLflow tracking: True
  - MLflow experiment: lstm_crypto_optimization

Starting APO Metaheuristic Optimization
Run ID: e829ed5e
MLflow run started: d3f261bed5734417ae34de33c6e75354

Evaluating initial population...
iter:-1 indv:13 fitness:0.0000 features:146 acc[tr:0.532 va:0.634 te:0.695]
iter:-1 indv:9 fitness:0.0000 features:127 acc[tr:0.532 va:0.635 te:0.695]
iter:-1 indv:1 fitness:0.0000 features:136 acc[tr:0.532 va:0.635 te:0.695]
iter:-1 indv:5 fitness:0.0000 features:120 acc[tr:0.463 va:0.366 te:0.305]
iter:-1 indv:3 fitness:0.0000 features:124 acc[tr:0.530 va:0.636 te:0.695]
iter:-1 indv:0 fitness:0.0000 features:12